# 09B. Experiments: threshold strategies for tp_sl_1_05**Цель.** Для зафиксированного target `tp_sl_1_05` и модели из шага 07 подобрать стратегии по probability-порогам и ширине HOLD-зоны, оценив net PnL и число сделок на тестовом дне.

## 1. Загрузка модели и данных

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import joblib

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in (
    '01_data_prep', '02_targets', '03_features', '04_models', '05_experiments'
) else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

model_path = os.path.join(_root, 'models', 'best_model.joblib')
bundle = joblib.load(model_path)
model = bundle['model']
scaler = bundle['scaler']
FEATURES = bundle['features']
TARGET_COL = bundle.get('target', 'target')
BEST_THRESHOLD = float(bundle.get('threshold', 0.6))

df = pd.read_parquet(os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet'))
valid = df.dropna(subset=FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]

sort_col = 'datetime' if 'datetime' in valid.columns else 'timestamp'
valid = valid.sort_values(['session_key', sort_col]).reset_index(drop=True)
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = valid.dropna(subset=['ret_next', 'volatility_14'])
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date
dates = sorted(valid['date'].unique())
test_date = dates[-1]

bt = valid[valid['date'] == test_date].copy().sort_values(['session_key', sort_col]).reset_index(drop=True)
X_bt = scaler.transform(bt[FEATURES].fillna(0))
proba = model.predict_proba(X_bt)[:, 1]

print('Model:', bundle.get('model_name', 'unknown'))
print('Val AUC:', bundle.get('val_auc', np.nan))
print('Test AUC:', bundle.get('test_auc', np.nan))
print('Test date:', test_date, 'bars:', len(bt))

## 2. Функция бектеста (как в шаге 08)

In [ ]:
def backtest_pnl(proba, ret, commission_rt=0.001, thr_hi=0.6, thr_lo=None, session_ids=None, hold_keep_prev=True):
    """Стратегия поверх proba.

    - long при proba >= thr_hi
    - short при proba <= thr_lo (если thr_lo None, берём 1-thr_hi, симметрично)
    - HOLD иначе.

    При HOLD по умолчанию сохраняем предыдущую позицию.
    Комиссия 0.1% round-trip (commission_rt) при смене позиции.
    """
    if thr_lo is None:
        thr_lo = 1 - thr_hi

    pred = np.full(len(proba), -1, dtype=int)
    pred[proba >= thr_hi] = 1
    pred[proba <= thr_lo] = 0
    mask_hold = (proba > thr_lo) & (proba < thr_hi)
    pred[mask_hold] = -99  # HOLD-маркер

    n = len(proba)
    pos = np.zeros(n, dtype=float)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i] = 1.0
            prev = 1.0
        elif pred[i] == 0:
            pos[i] = -1.0
            prev = -1.0
        else:  # HOLD
            pos[i] = prev if hold_keep_prev else 0.0

    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    if session_ids is not None:
        sess_chg = np.zeros(n, dtype=bool)
        sess_chg[1:] = (session_ids[1:] != session_ids[:-1])
        pos_prev = np.where(sess_chg, 0.0, pos_prev)

    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    n_changes = int(pos_changed.sum())
    fee_total = n_changes * (commission_rt / 2)

    pnl_raw = pos * ret
    pnl_net_sum = pnl_raw.sum() - fee_total

    return {
        'trades': n_changes,
        'gross_%': pnl_raw.sum() * 100,
        'net_%': pnl_net_sum * 100,
    }

## 3. Сетка стратегий и волатильностные корзины

In [ ]:
ret = bt['ret_next'].values
sessions = bt['session_key'].values
vol = bt['volatility_14'].values

# Разбиение на 3 корзины волатильности по квантилям
q1, q2 = np.quantile(vol, [0.33, 0.66])
vol_bucket = np.full(len(vol), 'mid', dtype=object)
vol_bucket[vol <= q1] = 'low'
vol_bucket[vol >= q2] = 'high'
bt['vol_bucket'] = vol_bucket

THR_HI_LIST = [0.55, 0.60, 0.65]
COMMISSION_RT = 0.001

rows = []
for thr_hi in THR_HI_LIST:
    thr_lo = 1.0 - thr_hi
    label = f"thr_hi={thr_hi:.2f}, thr_lo={thr_lo:.2f}"

    r_all = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, thr_hi=thr_hi, thr_lo=thr_lo, session_ids=sessions)
    rows.append({
        'scope': 'all',
        'thr_hi': thr_hi,
        'thr_lo': thr_lo,
        'label': label,
        'trades': r_all['trades'],
        'gross_%': r_all['gross_%'],
        'net_%': r_all['net_%'],
    })

    for bucket in ['low', 'mid', 'high']:
        mask = (bt['vol_bucket'].values == bucket)
        if mask.sum() < 100:
            continue
        r_b = backtest_pnl(proba[mask], ret[mask], commission_rt=COMMISSION_RT, thr_hi=thr_hi, thr_lo=thr_lo, session_ids=sessions[mask])
        rows.append({
            'scope': bucket,
            'thr_hi': thr_hi,
            'thr_lo': thr_lo,
            'label': label,
            'trades': r_b['trades'],
            'gross_%': r_b['gross_%'],
            'net_%': r_b['net_%'],
        })

res_df = pd.DataFrame(rows)
print('Top strategies (scope=all) by net_%:')
print(res_df[res_df['scope'] == 'all'].sort_values('net_%', ascending=False).to_string(index=False))

## 4. Сохранение результатов

In [ ]:
out_dir = os.path.join(_root, 'outputs')
os.makedirs(out_dir, exist_ok=True)
out_csv = os.path.join(out_dir, 'threshold_strategy_grid_tp_sl_1_05.csv')
res_df.to_csv(out_csv, index=False)
print('Saved:', out_csv)

## Итог шага 09B- Использована финальная модель `tp_sl_1_05 + LightGBM` из шага 07 (`models/best_model.joblib`).- На последнем дне протестированы стратегии с разными порогами proba и симметричной HOLD-зоной.- Для каждой стратегии посчитан net PnL и число сделок, в т.ч. по корзинам волатильности (low/mid/high).- Таблица результатов сохранена в `outputs/threshold_strategy_grid_tp_sl_1_05.csv` для дальнейшего анализа и выбора champion-стратегии.